In [5]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import gradio as gr
import json
import sqlite3

In [39]:
load_dotenv(override=True)  # Load environment variables from .env file
gemini_api_key = os.getenv("GEMINI_API_KEY")
if gemini_api_key is None:
    print("GEMINI_API_KEY is not set in the environment variables.")
else:
    print("GEMINI_API_KEY is set.")
Model="gemini-3.6-flash"
gemini_api_key = os.getenv("GEMINI_API_KEY")
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini=OpenAI(base_url=gemini_url, api_key=gemini_api_key)
DB="price.db"

GEMINI_API_KEY is set.


In [9]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [13]:
conn = sqlite3.connect(DB)

In [14]:
conn.execute("""
CREATE TABLE IF NOT EXISTS prices (
    city TEXT,
    price REAL
)
""")

In [15]:
conn.commit()

In [17]:
data = [
    ("London", 120),
    ("Paris", 100),
    ("New York", 150),
    ("India", 80),
    ("Vietnam", 70)
]
conn.executemany(
    "INSERT INTO prices (city, price) VALUES (?, ?)",
    data
)

conn.commit()

In [18]:
cursor = conn.execute("SELECT city, price FROM prices")

for row in cursor:
    print(row)

('London', 120.0)
('Paris', 100.0)
('New York', 150.0)
('India', 80.0)
('Vietnam', 70.0)


In [19]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [22]:
conn.execute("UPDATE prices SET city = LOWER(city)")
conn.commit()

In [23]:
cursor = conn.execute("SELECT city, price FROM prices")

for row in cursor:
    print(row)

('london', 120.0)
('paris', 100.0)
('new york', 150.0)
('india', 80.0)
('vietnam', 70.0)


In [24]:
get_ticket_price("Paris")

DATABASE TOOL CALLED: Getting price for Paris


'Ticket price to Paris is $100.0'

In [25]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": price_function}]
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [32]:

def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=Model, messages=messages)
    return response.choices[0].message.content
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [28]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=Model, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = gemini.chat.completions.create(model=Model, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [35]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [38]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/home/oman/AI_Learn/ai_projects/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 870, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/oman/AI_Learn/ai_projects/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 409, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/oman/AI_Learn/ai_projects/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2316, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/oman/AI_Learn/ai_projects/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1681, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/oman/AI_Learn/ai_projects/.venv/lib/python3.12/site-packages/gradio/utils.py", line 1081, 